# Qwen 2B LoRA for Text-to-SVG — v3 (Continue Training from v1)

This notebook:
1. Loads the v1 LoRA adapter (already trained on 50K SVGs)
2. Continues training on cleaned + normalized data with lower LR
3. Generates 1000 SVGs with batched inference

**Key fixes from v1 debugging:**
- `packing=False` (packing was causing extreme slowness)
- `text_tokenizer = tokenizer.tokenizer` (Unsloth wraps it in a VL processor)
- `eos_token_id` set to `<|im_end|>` tokens (prevents runaway generation)
- `input_ids` via `text_tokenizer.encode()` (avoids image processor errors)
- Decode only new tokens: `output_ids[0][input_ids.shape[1]:]`
- All SVGs forced to 256x256 canvas in post-processing

**Setup:** Runtime → Change runtime type → A100 GPU, High RAM

## 1. Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv

try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"

!uv pip install -qqq \
    "torch>=2.5" "triton>=3.1" {_numpy} {_pil} torchvision bitsandbytes xformers \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo"

!uv pip install --upgrade --no-deps tokenizers trl unsloth unsloth_zoo
!uv pip install datasets pandas lxml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Configuration

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"BF16: {torch.cuda.is_bf16_supported()}")

Torch: 2.10.0+cu128
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
BF16: True


In [ ]:
CONFIG = {
    # ═══ PATHS — UPDATE THESE ═══
    "train_csv": "/content/drive/MyDrive/train_clean.csv",
    "test_prompts_csv": "/content/drive/MyDrive/test.csv",
    "v1_adapter_path": "/content/drive/MyDrive/qwen2b_svg_lora",  # Your v1 LoRA adapter
    "output_dir": "/content/qwen2b_svg_lora_v5",
    "submission_path": "/content/submission_v5.csv",

    # Model — load from v1 adapter, NOT base model
    "max_seq_length": 2048,
    "load_in_4bit": True,

    # LoRA (must match v1 settings)
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0,

    # Training — lower LR for continued training
    "max_steps": 1500,
    "per_device_train_batch_size": 16,
    "gradient_accumulation_steps": 1,
    "learning_rate": 5e-5,           # Half of v1 to avoid catastrophic forgetting
    "warmup_steps": 50,
    "weight_decay": 0.01,
    "logging_steps": 10,
    "save_steps": 500,
    "eval_size": 0.01,

    # Inference
    "max_new_tokens": 2048,
    "max_svg_chars": 8000,
    "temperature": 0.3,
    "top_p": 0.85,
    "repetition_penalty": 1.1,
    "inference_batch_size": 8,
}

for k, v in CONFIG.items():
    print(f"  {k}: {v}")

  train_csv: /content/drive/MyDrive/train_clean.csv
  test_prompts_csv: /content/drive/MyDrive/test.csv
  v1_adapter_path: /content/drive/MyDrive/qwen2b_svg_lora
  output_dir: /content/qwen2b_svg_lora_v5
  submission_path: /content/submission_v5.csv
  max_seq_length: 2048
  load_in_4bit: True
  lora_r: 32
  lora_alpha: 64
  lora_dropout: 0
  max_steps: 1500
  per_device_train_batch_size: 16
  gradient_accumulation_steps: 1
  learning_rate: 5e-05
  warmup_steps: 50
  weight_decay: 0.01
  logging_steps: 10
  save_steps: 500
  eval_size: 0.01
  max_new_tokens: 2048
  max_svg_chars: 8000
  temperature: 0.3
  top_p: 0.85
  repetition_penalty: 1.1
  inference_batch_size: 8


## 3. Load, Clean, and Normalize Dataset

In [ ]:
df = pd.read_csv(CONFIG["train_csv"])
print(f"Raw CSV rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

Raw CSV rows: 33000
Columns: ['prompt', 'svg']


In [ ]:
# ── Basic cleaning ──
df = df.dropna(subset=["prompt", "svg"])
df["prompt"] = df["prompt"].astype(str).str.strip()
df["svg"] = df["svg"].astype(str).str.strip()
df = df[df["svg"].str.lower().str.startswith("<svg")]
df = df[df["prompt"].str.len() > 0]
print(f"After basic cleaning: {len(df)}")

After basic cleaning: 33000


In [ ]:
# ── Normalize all SVGs to 256x256 canvas ──
def normalize_svg_canvas(svg):
    svg = re.sub(r'width="[^"]*"', 'width="256"', svg, count=1)
    svg = re.sub(r'height="[^"]*"', 'height="256"', svg, count=1)
    svg = re.sub(r"width='[^']*'", "width='256'", svg, count=1)
    svg = re.sub(r"height='[^']*'", "height='256'", svg, count=1)
    svg = re.sub(r'viewBox="[^"]*"', 'viewBox="0 0 256 256"', svg, count=1)
    svg = re.sub(r"viewBox='[^']*'", "viewBox='0 0 256 256'", svg, count=1)
    if 'viewBox' not in svg:
        svg = svg.replace('<svg', '<svg viewBox="0 0 256 256"', 1)
    if 'width=' not in svg:
        svg = svg.replace('<svg', '<svg width="256" height="256"', 1)
    if 'xmlns=' not in svg:
        svg = svg.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    return svg


def is_valid_training_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    local_name = root.tag.split("}")[-1] if "}" in root.tag else root.tag
    return local_name.lower() == "svg"


df["svg"] = df["svg"].apply(normalize_svg_canvas)

df["valid"] = df["svg"].apply(is_valid_training_svg)
print(f"Valid SVGs after normalization: {df['valid'].sum()} / {len(df)}")
df = df[df["valid"]].drop(columns=["valid"])

df["svg_len"] = df["svg"].str.len()
df = df[(df["svg_len"] >= 100) & (df["svg_len"] <= 4000)]

print(f"After length filter (100-4000 chars): {len(df)}")
print(f"\nSVG length stats:")
print(df["svg_len"].describe())
print(f"\nPercentiles:")
print(df["svg_len"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

Valid SVGs after normalization: 33000 / 33000
After length filter (100-4000 chars): 28213

SVG length stats:
count    28213.000000
mean      1672.993585
std       1017.173599
min        118.000000
25%        810.000000
50%       1492.000000
75%       2427.000000
max       4000.000000
Name: svg_len, dtype: float64

Percentiles:
0.50    1492.0
0.75    2427.0
0.90    3223.0
0.95    3573.4
0.99    3911.0
Name: svg_len, dtype: float64


In [ ]:
# ── Format into chat template ──
SYSTEM_PROMPT = (
    "You are an SVG generation assistant. Given a text description, generate valid SVG code. "
    "Output ONLY the SVG markup. Always use: "
    '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">. '
    "Use only allowed elements: svg, g, path, rect, circle, ellipse, line, polyline, polygon, "
    "defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, "
    "title, desc, style, pattern, marker, filter. "
    "Keep the SVG under 8000 characters."
)


def format_sft_text(row):
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{row['prompt']}<|im_end|>\n"
        f"<|im_start|>assistant\n{row['svg']}<|im_end|>"
    )


df["text"] = df.apply(format_sft_text, axis=1)

df["est_tokens"] = df["text"].str.len() / 3.5
over_limit = (df["est_tokens"] > CONFIG["max_seq_length"]).sum()
print(f"Samples likely exceeding {CONFIG['max_seq_length']} tokens: {over_limit}")

Samples likely exceeding 2048 tokens: 0


In [ ]:
dataset = Dataset.from_pandas(df[["text"]].reset_index(drop=True))
dataset = dataset.shuffle(seed=SEED)

splits = dataset.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_dataset = splits["train"]
eval_dataset = splits["test"]

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

Train: 27930 | Eval: 283


## 4. Load v1 LoRA Adapter (Continue Training)

In [ ]:
from unsloth import FastLanguageModel

# Load the v1 adapter — this loads base model + LoRA weights together
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["v1_adapter_path"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=CONFIG["load_in_4bit"],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Loaded v1 adapter from: {CONFIG['v1_adapter_path']}")
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Loaded v1 adapter from: /content/drive/MyDrive/qwen2b_svg_lora
Trainable: 21,823,488 / 1,397,711,680 (1.56%)


## 5. Continue Training

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,                            # MUST be False — packing causes extreme slowness
    args=SFTConfig(
        output_dir=CONFIG["output_dir"],
        max_steps=CONFIG["max_steps"],
        per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        learning_rate=CONFIG["learning_rate"],
        warmup_steps=CONFIG["warmup_steps"],
        weight_decay=CONFIG["weight_decay"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=CONFIG["logging_steps"],
        eval_strategy="steps",
        eval_steps=500,
        save_steps=CONFIG["save_steps"],
        save_total_limit=2,
        report_to="none",
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        seed=SEED,
        dataloader_num_workers=4,
    ),
)

print(f"Trainer ready. max_steps={CONFIG['max_steps']}, lr={CONFIG['learning_rate']}")
print(f"Continuing from v1 adapter — loss should start ~0.36")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/27930 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/283 [00:00<?, ? examples/s]

Trainer ready. max_steps=1500, lr=5e-05
Continuing from v1 adapter — loss should start ~0.36


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1e9, 2)
print(f"GPU: {gpu_stats.name} | VRAM: {round(gpu_stats.total_memory/1e9,2)} GB | Reserved: {start_mem} GB")

GPU: NVIDIA A100-SXM4-80GB | VRAM: 85.09 GB | Reserved: 4.22 GB


In [ ]:
train_result = trainer.train()

used_mem = round(torch.cuda.max_memory_reserved() / 1e9, 2)
runtime = train_result.metrics['train_runtime'] / 60
print(f"\nDone! {runtime:.1f} min | Peak VRAM: {used_mem} GB")
print(f"Final train loss: {train_result.metrics.get('train_loss', 'N/A')}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 27,930 | Num Epochs = 1 | Total steps = 1,500
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 21,823,488 of 2,235,065,152 (0.98% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
500,0.320503,0.306181
1000,0.286331,0.299891
1500,0.278727,0.297768



Done! 202.1 min | Peak VRAM: 51.38 GB
Final train loss: 0.29089773082733156


In [ ]:
# ── Save v3 adapter ──
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"v3 adapter saved to: {CONFIG['output_dir']}")

# Also save to Drive immediately
import shutil
drive_path = "/content/drive/MyDrive/qwen2b_svg_lora_v3"
shutil.copytree(CONFIG["output_dir"], drive_path, dirs_exist_ok=True)
print(f"Backed up to Drive: {drive_path}")

v3 adapter saved to: /content/qwen2b_svg_lora_v5
Backed up to Drive: /content/drive/MyDrive/qwen2b_svg_lora_v3


## 6. Inference Setup

**Important notes from v1 debugging:**
- Must use `tokenizer.tokenizer` to get the actual text tokenizer (Unsloth wraps it in Qwen3VLProcessor)
- Must use `text_tokenizer.encode()` not `tokenizer()` to avoid image processor errors
- Must set `eos_token_id` to `<|im_end|>` token IDs or generation runs to max_new_tokens every time
- Must decode only new tokens via `output_ids[0][input_len:]` to avoid echoing the prompt

In [ ]:
# ── SVG utilities ──
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line", "polyline", "polygon",
    "defs", "use", "symbol", "clipPath", "mask", "linearGradient", "radialGradient",
    "stop", "text", "tspan", "title", "desc", "style", "pattern", "marker", "filter",
    "clippath", "lineargradient", "radialgradient",
}


def extract_svg(text):
    m = SVG_REGEX.search(text)
    if not m:
        return ""
    svg = m.group(0).strip()
    if len(svg) > CONFIG["max_svg_chars"]:
        return ""
    return svg


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    local_name = root.tag.split("}")[-1] if "}" in root.tag else root.tag
    if local_name.lower() != "svg":
        return False
    for elem in root.iter():
        tag = elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag
        if tag.lower() not in ALLOWED_TAGS:
            return False
    return True


def fix_svg_canvas(svg):
    """Force 256x256 canvas."""
    svg = re.sub(r'width="[^"]*"', 'width="256"', svg, count=1)
    svg = re.sub(r'height="[^"]*"', 'height="256"', svg, count=1)
    svg = re.sub(r"width='[^']*'", "width='256'", svg, count=1)
    svg = re.sub(r"height='[^']*'", "height='256'", svg, count=1)
    svg = re.sub(r'viewBox="[^"]*"', 'viewBox="0 0 256 256"', svg, count=1)
    svg = re.sub(r"viewBox='[^']*'", "viewBox='0 0 256 256'", svg, count=1)
    if 'viewBox' not in svg:
        svg = svg.replace('<svg', '<svg viewBox="0 0 256 256"', 1)
    if 'width=' not in svg:
        svg = svg.replace('<svg', '<svg width="256" height="256"', 1)
    if 'xmlns=' not in svg:
        svg = svg.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    return svg


def repair_truncated_svg(decoded):
    """Try to close a truncated SVG that's missing </svg>."""
    if "<svg" not in decoded:
        return ""
    truncated = decoded.strip()
    if "</svg>" in truncated:
        return ""
    # Close any dangling attribute/quote
    truncated = truncated.rstrip('"\' =')
    # Track open container tags
    open_tags = []
    for m in re.finditer(r'<(g|clipPath|mask|defs|linearGradient|radialGradient|pattern|filter|symbol|text)[ >]', truncated):
        open_tags.append(m.group(1))
    for m in re.finditer(r'</(g|clipPath|mask|defs|linearGradient|radialGradient|pattern|filter|symbol|text)>', truncated):
        if open_tags and open_tags[-1] == m.group(1):
            open_tags.pop()
    # Close current element + open containers + svg
    truncated += '"/>'
    for tag in reversed(open_tags):
        truncated += f'</{tag}>'
    truncated += '</svg>'
    return extract_svg(truncated)


def fallback_svg(prompt):
    """Keyword-aware fallback SVG."""
    prompt_lower = prompt.lower()

    colors = {
        "red": "#FF0000", "blue": "#0000FF", "green": "#008000",
        "yellow": "#FFD700", "orange": "#FFA500", "purple": "#800080",
        "black": "#000000", "white": "#FFFFFF", "pink": "#FFC0CB",
        "gray": "#808080", "grey": "#808080", "brown": "#8B4513",
        "cyan": "#00FFFF", "magenta": "#FF00FF", "gold": "#FFD700",
    }
    fill = "#333333"
    for name, hex_val in colors.items():
        if name in prompt_lower:
            fill = hex_val
            break

    if any(w in prompt_lower for w in ["circle", "round", "dot", "ball", "sun", "moon", "ring"]):
        shape = f'<circle cx="128" cy="128" r="80" fill="{fill}"/>'
    elif any(w in prompt_lower for w in ["square", "box", "rect", "frame", "window"]):
        shape = f'<rect x="48" y="48" width="160" height="160" fill="{fill}" rx="8"/>'
    elif any(w in prompt_lower for w in ["triangle", "arrow", "play"]):
        shape = f'<polygon points="128,30 220,220 36,220" fill="{fill}"/>'
    elif any(w in prompt_lower for w in ["star", "favorite"]):
        shape = f'<polygon points="128,20 148,100 230,100 162,148 186,228 128,178 70,228 94,148 26,100 108,100" fill="{fill}"/>'
    elif any(w in prompt_lower for w in ["heart", "love", "like"]):
        shape = f'<path d="M128,224 C80,180 20,140 20,90 C20,50 50,20 90,20 C110,20 128,35 128,35 C128,35 146,20 166,20 C206,20 236,50 236,90 C236,140 176,180 128,224Z" fill="{fill}"/>'
    elif any(w in prompt_lower for w in ["diamond", "rhombus"]):
        shape = f'<polygon points="128,20 230,128 128,236 26,128" fill="{fill}"/>'
    elif any(w in prompt_lower for w in ["cross", "plus", "add", "medical"]):
        shape = f'<path d="M98,28 h60 v70 h70 v60 h-70 v70 h-60 v-70 h-70 v-60 h70 z" fill="{fill}"/>'
    elif any(w in prompt_lower for w in ["hexagon"]):
        shape = f'<polygon points="128,28 210,68 210,188 128,228 46,188 46,68" fill="{fill}"/>'
    else:
        shape = f'<rect x="38" y="38" width="180" height="180" rx="20" fill="{fill}"/>'

    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect width="256" height="256" fill="white"/>'
        f'{shape}'
        '</svg>'
    )


print("SVG utilities loaded.")

SVG utilities loaded.


In [ ]:
# ── Prepare tokenizer for inference ──
# IMPORTANT: Unsloth wraps tokenizer in Qwen3VLProcessor.
# We need the inner tokenizer for text-only generation.
FastLanguageModel.for_inference(model)

text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, 'tokenizer') else tokenizer
text_tokenizer.padding_side = "left"
if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token

# IMPORTANT: Get <|im_end|> token IDs for early stopping
eos_ids = text_tokenizer.encode("<|im_end|>", add_special_tokens=False)
print(f"EOS token ids: {eos_ids}")
print(f"Pad token: {text_tokenizer.pad_token}")
print("Inference ready.")

EOS token ids: [248046]
Pad token: <|vision_pad|>
Inference ready.


In [ ]:
# ── Single prompt generation ──
def generate_svg(prompt, max_new_tokens=None):
    if max_new_tokens is None:
        max_new_tokens = CONFIG["max_new_tokens"]

    chat_text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    # IMPORTANT: use text_tokenizer.encode(), NOT tokenizer() — avoids image processor
    input_ids = text_tokenizer.encode(
        chat_text, return_tensors="pt", add_special_tokens=False
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=CONFIG["temperature"],
            top_p=CONFIG["top_p"],
            repetition_penalty=CONFIG["repetition_penalty"],
            eos_token_id=eos_ids,         # IMPORTANT: stops at <|im_end|>
        )

    # IMPORTANT: decode only NEW tokens, skip the input prompt
    new_tokens = output_ids[0][input_ids.shape[1]:]
    decoded = text_tokenizer.decode(new_tokens, skip_special_tokens=True)

    svg = extract_svg(decoded)
    if not svg:
        svg = repair_truncated_svg(decoded)
    if svg:
        svg = fix_svg_canvas(svg)
    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)

    return svg


# ── Batched generation ──
def generate_svgs_batched(prompts, max_new_tokens=None, batch_size=None):
    if max_new_tokens is None:
        max_new_tokens = CONFIG["max_new_tokens"]
    if batch_size is None:
        batch_size = CONFIG["inference_batch_size"]

    results = []
    t_start = time.time()

    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]

        chat_texts = [
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{p}<|im_end|>\n"
            f"<|im_start|>assistant\n"
            for p in batch_prompts
        ]

        # IMPORTANT: use text_tokenizer for batched tokenization
        inputs = text_tokenizer(
            chat_texts, return_tensors="pt", padding=True, add_special_tokens=False
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=CONFIG["temperature"],
                top_p=CONFIG["top_p"],
                repetition_penalty=CONFIG["repetition_penalty"],
                eos_token_id=eos_ids,
            )

        input_len = inputs["input_ids"].shape[1]
        for j, ids in enumerate(output_ids):
            new_tokens = ids[input_len:]
            decoded = text_tokenizer.decode(new_tokens, skip_special_tokens=True)

            svg = extract_svg(decoded)
            if not svg:
                svg = repair_truncated_svg(decoded)
            if svg:
                svg = fix_svg_canvas(svg)
            if not is_valid_svg(svg):
                svg = fallback_svg(batch_prompts[j])
            results.append(svg)

        elapsed = time.time() - t_start
        done = len(results)
        rate = done / max(elapsed, 0.01)
        remaining = (len(prompts) - done) / max(rate, 0.01) / 60
        fallbacks = sum(1 for s in results if '<rect width="256" height="256" fill="white"/>' in s)
        print(
            f"  [{done}/{len(prompts)}] {rate:.2f} samples/s, "
            f"~{remaining:.1f} min left, {fallbacks} fallbacks",
            flush=True,
        )

    return results


print("Generation functions loaded.")

Generation functions loaded.


In [ ]:
# ── Sanity check (use small max_new_tokens for speed) ──
print("Sanity check:")
test_cases = ["a red circle", "a green tree with brown trunk", "a blue star icon", "a black cat silhouette"]
for p in test_cases:
    svg = generate_svg(p, max_new_tokens=512)
    print(f"{p} → Valid: {is_valid_svg(svg)}, Length: {len(svg)} chars")
    print(f"  {svg[:200]}...\n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Sanity check:
a red circle → Valid: True, Length: 326 chars
  <svg fill="none" height="256" viewBox="0 0 256 256" width="256" xmlns="http://www.w3.org/2000/svg"><path d="m12 4c-5.52 0-10 4.48-10 10s4.48 10 10 10 10-4.48 10-10-4.48-10-10-10zm0 18c-4.97 0-9-4.03-9...

a green tree with brown trunk → Valid: True, Length: 715 chars
  <svg fill="none" height="256" viewBox="0 0 256 256" width="256" xmlns="http://www.w3.org/2000/svg"><path d="m17.94 17.94c-.28-.28-.75-.28-1.03 0l-4.24 4.24h-4.24v-4.24c0-.28-.28-.75-.75-.75s-.75.28-.7...

a blue star icon → Valid: True, Length: 526 chars
  <svg fill="none" height="256" viewBox="0 0 256 256" width="256" xmlns="http://www.w3.org/2000/svg"><path d="m19.47 12c-.18-2.65-1.84-4.87-4.47-5.87l-1.41-.67c-2.63-1.29-5.73-.29-7.42 2.4s-.29 5.73 2.4...

a black cat silhouette → Valid: True, Length: 298 chars
  <svg fill="none" height="256" viewBox="0 0 256 256" width="256" xmlns="http://www.w3.org/2000/svg"><path d="m19 14c-1.1 0-2-.9-2-2s.9-2 2-2 2 .9 2 2

In [ ]:
# ── Debug: check raw output for one prompt ──
debug_prompt = "a red circle"
chat_text = (
    f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
    f"<|im_start|>user\n{debug_prompt}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)
input_ids = text_tokenizer.encode(chat_text, return_tensors="pt", add_special_tokens=False).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids=input_ids,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.3,
        top_p=0.85,
        eos_token_id=eos_ids,
    )

new_tokens = output_ids[0][input_ids.shape[1]:]
print(f"Generated {len(new_tokens)} tokens")
raw = text_tokenizer.decode(new_tokens, skip_special_tokens=False)
print(f"Raw output:\n{raw[:1000]}")

Generated 256 tokens
Raw output:
<svg height="256" viewBox="0 0 256 256" width="256" xmlns="http://www.w3.org/2000/svg"><path d="m10 22c-5.52 0-10-4.48-10-10s4.48-10 10-10 10 4.48 10 10-4.48 10-10 10zm0-18c-4.42 0-8 3.58-8 8s3.58 8 8 8 8-3.58 8-8-3.58-8-8-8z" fill="#ff4444"/><path d="m10 22c-5.52 0-10-4.48-10-10s4.48-10 10-10 10 4.48 10 10-4.48 10-10 10


## 7. Generate Submission

In [ ]:
test_df = pd.read_csv(CONFIG["test_prompts_csv"])
print(f"Test prompts: {len(test_df)}")
test_df.head()

Test prompts: 1000


,id,prompt
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,firewood stack cut logs wood with leaf illustr...
1,6eede943219547c22ac56085027d33cc,The image shows five horizontal lines of varyi...
2,ea045c7a247166f061ce504d9b7ccaab,A stylized icon depicting a curved arrow withi...
3,8fe82f3af89e487b31236ca829c3f071,The image contains black geometric shapes agai...
4,600464e4d92c75338462271a09b3f176,The image shows a single dark gray triangle po...


In [ ]:
# ── Generate all SVGs ──
all_prompts = test_df["prompt"].tolist()
all_svgs = generate_svgs_batched(all_prompts)

  [8/1000] 0.03 samples/s, ~628.3 min left, 0 fallbacks
  [16/1000] 0.03 samples/s, ~602.8 min left, 0 fallbacks
  [24/1000] 0.03 samples/s, ~588.2 min left, 0 fallbacks
  [32/1000] 0.03 samples/s, ~580.3 min left, 0 fallbacks
  [40/1000] 0.03 samples/s, ~572.8 min left, 0 fallbacks
  [48/1000] 0.03 samples/s, ~548.0 min left, 0 fallbacks
  [56/1000] 0.03 samples/s, ~544.4 min left, 0 fallbacks
  [64/1000] 0.03 samples/s, ~540.1 min left, 0 fallbacks
  [72/1000] 0.03 samples/s, ~536.1 min left, 0 fallbacks
  [80/1000] 0.03 samples/s, ~531.7 min left, 0 fallbacks
  [88/1000] 0.03 samples/s, ~527.4 min left, 0 fallbacks
  [96/1000] 0.03 samples/s, ~512.8 min left, 0 fallbacks
  [104/1000] 0.03 samples/s, ~510.8 min left, 0 fallbacks
  [112/1000] 0.03 samples/s, ~507.0 min left, 0 fallbacks
  [120/1000] 0.03 samples/s, ~503.1 min left, 0 fallbacks
  [128/1000] 0.03 samples/s, ~499.3 min left, 0 fallbacks
  [136/1000] 0.03 samples/s, ~495.3 min left, 0 fallbacks
  [144/1000] 0.03 samples/s

KeyboardInterrupt: 

In [ ]:
print(all_svgs)

NameError: name 'all_svgs' is not defined

In [ ]:
# ── Build and validate submission ──
rows = []
invalid_count = 0

for idx, (_, row) in enumerate(test_df.iterrows()):
    svg = all_svgs[idx]
    if len(svg) > CONFIG["max_svg_chars"] or not is_valid_svg(svg):
        svg = fallback_svg(row["prompt"])
        invalid_count += 1
    rows.append({"id": row["id"], "svg": svg})

sub_df = pd.DataFrame(rows)
sub_df["svg_len"] = sub_df["svg"].str.len()

print(f"SVG length stats:")
print(sub_df["svg_len"].describe())
print(f"\nFallbacks: {invalid_count} / {len(sub_df)} ({100*invalid_count/len(sub_df):.1f}%)")
print(f"Max SVG length: {sub_df['svg_len'].max()}")

sub_df[["id", "svg"]].to_csv(CONFIG["submission_path"], index=False)
print(f"\nSaved: {CONFIG['submission_path']}")

NameError: name 'all_svgs' is not defined

In [ ]:
from google.colab import files
files.download(CONFIG["submission_path"])

## 8. Save Model

In [ ]:
# Push to HuggingFace
PUSH_TO_HF = False

if PUSH_TO_HF:
    HF_USERNAME = "YOUR_USERNAME"
    HF_TOKEN = "YOUR_HF_TOKEN"
    model.push_to_hub(f"{HF_USERNAME}/qwen2b-svg-lora-v3", token=HF_TOKEN)
    tokenizer.push_to_hub(f"{HF_USERNAME}/qwen2b-svg-lora-v3", token=HF_TOKEN)
    print("Pushed to HuggingFace Hub.")

In [ ]:
# Save merged model for Kaggle offline inference
SAVE_MERGED = True

if SAVE_MERGED:
    merged_dir = "/content/qwen2b_svg_merged_v3"
    model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
    print(f"Merged model: {merged_dir}")
    print("Upload as Kaggle dataset for offline inference.")

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 1 files from cache to `/content/qwen2b_svg_merged_v3`:   0%|          | 0/1 [00:00<?, ?it/s]
Unsloth: Copying 1 files from cache to `/content/qwen2b_svg_merged_v3`: 100%|██████████| 1/1 [00:04<00:00,  4.15s/it]


Successfully copied all 1 files from cache to `/content/qwen2b_svg_merged_v3`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 8594.89it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:18<00:00, 18.49s/it]


Unsloth: Merge process complete. Saved to `/content/qwen2b_svg_merged_v3`
Merged model: /content/qwen2b_svg_merged_v3
Upload as Kaggle dataset for offline inference.


In [ ]:
import shutil

# Source folder in Colab
source_dir = '/content/qwen2b_svg_merged_v3'
# Destination folder in your Google Drive
destination_dir = '/content/drive/MyDrive/qwen2b_svg_lora_v5'

# Copy the entire folder recursively
shutil.copytree(source_dir, destination_dir)
print("Folder successfully copied to Google Drive!")

Folder successfully copied to Google Drive!
